## CardioGuard — Data Preprocessing

-  Fix  data quality issues and prepare clean datasets
- for feature engineering and modelling

###  Setup & import libraries

In [29]:
import os
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt

warnings.filterwarnings("ignore")

# Define path

DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")

os.makedirs(DATA_PROCESSED, exist_ok=True)


### STEP 1 - Load the datasets

In [17]:

demo = pd.read_sas(os.path.join(DATA_RAW, "DEMO_J.XPT"))
bmx = pd.read_sas(os.path.join(DATA_RAW, "BMX_J.XPT"))
bpx = pd.read_sas(os.path.join(DATA_RAW, "BPX_J.XPT"))
glu = pd.read_sas(os.path.join(DATA_RAW, "GLU_J.XPT"))
ghb = pd.read_sas(os.path.join(DATA_RAW, "GHB_J.XPT"))
tchol = pd.read_sas(os.path.join(DATA_RAW, "TCHOL_J.XPT"))
hdl = pd.read_sas(os.path.join(DATA_RAW, "HDL_J.XPT"))
trigly = pd.read_sas(os.path.join(DATA_RAW, "TRIGLY_J.XPT"))
smq = pd.read_sas(os.path.join(DATA_RAW, "SMQ_J.XPT"))
paq = pd.read_sas(os.path.join(DATA_RAW, "PAQ_J.XPT"))
bpq = pd.read_sas(os.path.join(DATA_RAW, "BPQ_J.XPT"))
diq = pd.read_sas(os.path.join(DATA_RAW, "DIQ_J.XPT"))

In [18]:
#  - Standardize column names

datasets = {
    "demo": demo,
    "bmx": bmx,
    "bpx": bpx,
    "glu": glu,
    "ghb": ghb,
    "tchol": tchol,
    "hdl": hdl,
    "trigly": trigly,
    "smq": smq,
    "paq": paq,
    "bpq": bpq,
    "diq": diq
}

for name, df in datasets.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.lower()
    )

### STEP 2 - Select Relevant Column

In [19]:

demo = demo[["seqn","riagendr","ridageyr","ridreth3","dmdeduc2","indfmpir"]]
bmx = bmx[["seqn","bmxbmi","bmxwaist","bmxwt","bmxht"]]
bpx = bpx[["seqn","bpxsy1","bpxdi1"]]
glu = glu[["seqn","lbxglu"]]
ghb = ghb[["seqn","lbxgh"]]
tchol = tchol[["seqn","lbxtc"]]
hdl = hdl[["seqn","lbdhdd"]]
trigly = trigly[["seqn","lbxtr","lbdldl"]]
smq = smq[["seqn","smq020"]]
paq = paq[["seqn","paq650","paq665"]]
bpq = bpq[["seqn","bpq020","bpq040a"]]
diq = diq[["seqn","diq010"]]

### STEP 3 - Rename columns for readability

In [20]:
# Rename participant ID across all datasets
datasets = [
    demo, bmx, bpx, glu, ghb,
    tchol, hdl, trigly,
    smq, paq, bpq, diq
]

for df in datasets:
    df.rename(columns={"seqn": "participant_id"}, inplace=True)

# Demographics
demo.rename(columns={
    "riagendr": "gender",
    "ridageyr": "age",
    "ridreth3": "ethnicity",
    "dmdeduc2": "education_level",
    "indfmpir": "poverty_income_ratio"
}, inplace=True)

# Body Measures
bmx.rename(columns={
    "bmxbmi": "bmi",
    "bmxwaist": "waist_circumference",
    "bmxwt": "weight",
    "bmxht": "height"
}, inplace=True)

# Blood Pressure
bpx.rename(columns={
    "bpxsy1": "systolic_bp",
    "bpxdi1": "diastolic_bp"
}, inplace=True)

# Glucose
glu.rename(columns={
    "lbxglu": "fasting_glucose"
}, inplace=True)

# HbA1c
ghb.rename(columns={
    "lbxgh": "hba1c"
}, inplace=True)

# Lipids
tchol.rename(columns={
    "lbxtc": "total_cholesterol"
}, inplace=True)

hdl.rename(columns={
    "lbdhdd": "hdl_cholesterol"
}, inplace=True)

trigly.rename(columns={
    "lbxtr": "triglycerides",
    "lbdldl": "ldl_cholesterol"
}, inplace=True)

# Smoking
smq.rename(columns={
    "smq020": "smoking_status"
}, inplace=True)

# Physical Activity
paq.rename(columns={
    "paq650": "vigorous_activity",
    "paq665": "moderate_activity"
}, inplace=True)

# Blood Pressure Questionnaire
bpq.rename(columns={
    "bpq020": "hypertension_history",
    "bpq040a": "bp_medication"
}, inplace=True)

# Diabetes Questionnaire
diq.rename(columns={
    "diq010": "diabetes_history"
}, inplace=True)

### STEP 4 - Map categorical values

In [21]:
gender_map = {
    1: "Male",
    2: "Female"
}

ethnicity_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other Race"
}

education_map = {
    1: "Less than 9th grade",
    2: "9th-11th grade",
    3: "High school graduate",
    4: "Some college",
    5: "College graduate or above",
    7: np.nan,
    9: np.nan
}

yes_no_map = {
    1: "Yes",
    2: "No",
    7: np.nan,
    9: np.nan
}

diabetes_map = {
    1: "Yes",
    2: "No",
    3: "Borderline",
    7: np.nan,
    9: np.nan
}
# Apply mappings

demo["gender"] = demo["gender"].map(gender_map)
demo["ethnicity"] = demo["ethnicity"].map(ethnicity_map)
demo["education_level"] = demo["education_level"].map(education_map)

smq["smoking_status"] = smq["smoking_status"].map(yes_no_map)

paq["vigorous_activity"] = paq["vigorous_activity"].map(yes_no_map)
paq["moderate_activity"] = paq["moderate_activity"].map(yes_no_map)

bpq["hypertension_history"] = bpq["hypertension_history"].map(yes_no_map)
bpq["bp_medication"] = bpq["bp_medication"].map(yes_no_map)

diq["diabetes_history"] = diq["diabetes_history"].map(diabetes_map)

### STEP 5 - Handle missingness

In [23]:
# Step 5a: Convert NHANES special codes to NaN


# 7  = Refused
# 9  = Don't know
# 77 = Refused
# 99 = Don't know

for df in [
    demo,bmx,bpx,glu,ghb,
    tchol,hdl,trigly,
    smq,paq,bpq,diq
]:
    
    df.replace(
        [7,9,77,99,777,999],
        np.nan,
        inplace=True
    )



In [26]:
# Step 5b : Check Missingness


for df in datasets:

    print(f"\n{name}")

    missing_pct = (
        df.isnull().sum()
        / len(df)
        * 100
    ).round(2)

    print(missing_pct)


diq
participant_id           0.00
gender                   0.00
age                      4.54
ethnicity                0.00
education_level         39.96
poverty_income_ratio    13.30
dtype: float64

diq
participant_id          0.00
bmi                     8.03
waist_circumference    13.12
weight                  1.76
height                  7.94
dtype: float64

diq
participant_id     0.0
systolic_bp       27.6
diastolic_bp      27.6
dtype: float64

diq
participant_id     0.00
fasting_glucose    8.47
dtype: float64

diq
participant_id    0.00
hba1c             6.11
dtype: float64

diq
participant_id       0.00
total_cholesterol    9.46
dtype: float64

diq
participant_id     0.00
hdl_cholesterol    9.89
dtype: float64

diq
participant_id     0.00
triglycerides      8.10
ldl_cholesterol    9.22
dtype: float64

diq
participant_id     0.00
smoking_status    12.91
dtype: float64

diq
participant_id       0.0
vigorous_activity    0.0
moderate_activity    0.0
dtype: float64

diq
participant_

### STEP 7 — Check Outliers

In [27]:
 # we inspect before treating outliers.

for df in [
    bmx,bpx,glu,
    ghb,tchol,hdl,trigly
]:
    
    print(df.describe())
    print(df.skew())

       participant_id          bmi  waist_circumference       weight  \
count     8704.000000  8005.000000          7562.000000  8551.000000   
mean     98315.452091    26.577502            89.937345    65.129622   
std       2669.112899     8.260724            22.849881    32.881481   
min      93703.000000    12.300000            40.000000     3.200000   
25%      96000.750000    20.400000            73.800000    43.100000   
50%      98308.500000    25.800000            91.200000    67.700000   
75%     100625.250000    31.300000           105.400000    85.500000   
max     102956.000000    86.200000           169.500000   242.600000   

            height  
count  8013.000000  
mean    156.614963  
std      22.234102  
min      78.300000  
25%     151.400000  
50%     161.900000  
75%     171.200000  
max     197.700000  
participant_id         0.006326
bmi                    0.936349
waist_circumference    0.057707
weight                 0.151661
height                -1.360542
dt

In [32]:
# STEP 7c - Basic numeric cleaning

# Manage Impossible Values

# we ensure numeric values are not negative. 

#  without touching genuine clinical extremes.

# example:

# BMI < 0
# Age < 0
# Glucose < 0
# Cholesterol < 0



numeric_datasets = [demo, bmx, bpx, glu, ghb, tchol, hdl, trigly]

for df in numeric_datasets:
    for col in df.select_dtypes(include=["int64", "float64"]).columns:
        if col != "participant_id":
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].clip(lower=0)

### STEP 8 — Remove Duplicates

In [33]:
for df in [
    demo,bmx,bpx,glu,ghb,
    tchol,hdl,trigly,
    smq,paq,bpq,diq
]:
    
    df.drop_duplicates(inplace=True)


### STEP 9 — Validate Missing Values


In [34]:

for name, df in {
    "demo":demo,
    "bmx":bmx,
    "bpx":bpx,
    "glu":glu,
    "ghb":ghb,
    "tchol":tchol,
    "hdl":hdl,
    "trigly":trigly,
    "smq":smq,
    "paq":paq,
    "bpq":bpq,
    "diq":diq
}.items():

    print(f"\n{name}")
    print(df.isnull().sum())






demo
participant_id             0
gender                     0
age                      420
ethnicity                  0
education_level         3698
poverty_income_ratio    1231
dtype: int64

bmx
participant_id            0
bmi                     699
waist_circumference    1142
weight                  153
height                  691
dtype: int64

bpx
participant_id       0
systolic_bp       2402
diastolic_bp      2402
dtype: int64

glu
participant_id       0
fasting_glucose    257
dtype: int64

ghb
participant_id      0
hba1c             391
dtype: int64

tchol
participant_id         0
total_cholesterol    703
dtype: int64

hdl
participant_id       0
hdl_cholesterol    735
dtype: int64

trigly
participant_id       0
triglycerides      246
ldl_cholesterol    280
dtype: int64

smq
participant_id      0
smoking_status    868
dtype: int64

paq
participant_id       0
vigorous_activity    0
moderate_activity    0
dtype: int64

bpq
participant_id             0
hypertension_history      10


### STEP 10 — Save Clean Datasets

In [35]:
files = {
    "demo_clean.csv": demo,
    "bmx_clean.csv": bmx,
    "bpx_clean.csv": bpx,
    "glu_clean.csv": glu,
    "ghb_clean.csv": ghb,
    "tchol_clean.csv": tchol,
    "hdl_clean.csv": hdl,
    "trigly_clean.csv": trigly,
    "smq_clean.csv": smq,
    "paq_clean.csv": paq,
    "bpq_clean.csv": bpq,
    "diq_clean.csv": diq
}

for fname, df in files.items():
    
    df.to_csv(
        os.path.join(DATA_PROCESSED, fname),
        index=False
    )

    print(f"{fname} saved successfully")

demo_clean.csv saved successfully
bmx_clean.csv saved successfully
bpx_clean.csv saved successfully
glu_clean.csv saved successfully
ghb_clean.csv saved successfully
tchol_clean.csv saved successfully
hdl_clean.csv saved successfully
trigly_clean.csv saved successfully
smq_clean.csv saved successfully
paq_clean.csv saved successfully
bpq_clean.csv saved successfully
diq_clean.csv saved successfully
